In [1]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os

In [2]:
with_mask_dir = 'D:/AST_DANCE/DEEP_LEARNING/data/with_mask'
without_mask_dir = 'D:/AST_DANCE/DEEP_LEARNING/data/without_mask'
with_mask_files = os.listdir(with_mask_dir)
without_mask_files = os.listdir(without_mask_dir)

In [3]:
data = []
labels = []

In [4]:
for image_file in with_mask_files:
    image_path = os.path.join(with_mask_dir, image_file)
    try:
        image = Image.open(image_path)
        image = image.resize((128, 128))
        image = image.convert('RGB')
        image_array = np.array(image)
        data.append(image_array)
        labels.append(1)
    except Exception as e:
        continue

c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\PIL\Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


In [5]:
for image_file in without_mask_files:
    image_path = os.path.join(without_mask_dir, image_file)
    try:
        image = Image.open(image_path)
        image = image.resize((128, 128))
        image = image.convert('RGB')
        image_array = np.array(image)
        data.append(image_array)
        labels.append(0)
    except Exception as e:
        continue

In [6]:
X = np.array(data)
y = np.array(labels)

In [7]:
X_scaled = X/255.0

In [8]:
print("shape of X (images array) : ", X_scaled.shape)
print("shape of y (labels array) : ", y.shape)
print("sample label values (first 5 vs last 5) : ", y[:5], y[-5:])

shape of X (images array) :  (7553, 128, 128, 3)
shape of y (labels array) :  (7553,)
sample label values (first 5 vs last 5) :  [1 1 1 1 1] [0 0 0 0 0]


In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print("X_train shape (Training images) : ", X_train.shape)
print("X_test shape (Testing images) : ", X_test.shape)
print("y_train shape (Training labels) : ", y_train.shape)
print("y_test shape (Testing labels) : ", y_test.shape)

X_train shape (Training images) :  (6042, 128, 128, 3)
X_test shape (Testing images) :  (1511, 128, 128, 3)
y_train shape (Training labels) :  (6042,)
y_test shape (Testing labels) :  (1511,)


In [10]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
num_of_classes = 2

model = keras.Sequential([
    layers.Conv2D(32, kernel_size=(3, 3), activation="relu", input_shape=(128, 128, 3)),
    layers.MaxPooling2D(pool_size=(2, 2)),
    
    layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),
    layers.MaxPooling2D(pool_size=(2, 2)),
    
    layers.Flatten(),
    
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.5),
    
    layers.Dense(num_of_classes, activation="sigmoid")
])

c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
model.compile(
    optimizer = "adam",
    loss = "sparse_categorical_crossentropy",
    metrics = ["accuracy"]
)

In [12]:
history = model.fit(
    X_train, y_train, validation_split = 0.1, epochs = 5
)

Epoch 1/5
170/170 ━━━━━━━━━━━━━━━━━━━━ 81s 421ms/step - accuracy: 0.7988 - loss: 0.4674 - val_accuracy: 0.9058 - val_loss: 0.2262
Epoch 2/5
170/170 ━━━━━━━━━━━━━━━━━━━━ 79s 466ms/step - accuracy: 0.8902 - loss: 0.2876 - val_accuracy: 0.9273 - val_loss: 0.1825
Epoch 3/5
170/170 ━━━━━━━━━━━━━━━━━━━━ 65s 384ms/step - accuracy: 0.9066 - loss: 0.2303 - val_accuracy: 0.9289 - val_loss: 0.1811
Epoch 4/5
170/170 ━━━━━━━━━━━━━━━━━━━━ 55s 320ms/step - accuracy: 0.9285 - loss: 0.1905 - val_accuracy: 0.9355 - val_loss: 0.1742
Epoch 5/5
170/170 ━━━━━━━━━━━━━━━━━━━━ 74s 434ms/step - accuracy: 0.9373 - loss: 0.1554 - val_accuracy: 0.9471 - val_loss: 0.1551


In [13]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)

print(f"test loss : {test_loss:.4f}")
print(f"test accuarcy : {test_accuracy*100:.2f}%")

48/48 ━━━━━━━━━━━━━━━━━━━━ 6s 120ms/step - accuracy: 0.9365 - loss: 0.2051
test loss : 0.2051
test accuarcy : 93.65%


In [ ]:
def predict_face(image_path, model):
    try:
        raw_image = Image.open(image_path)
        resized_image = raw_image.resize((128, 128))
        rgb_image = resized_image.convert('RGB')
        image_array = np.array(rgb_image)
        scaled_image = image_array/255.0
        input_tensor = np.reshape(scaled_image, (1, 128, 128, 3))
        
        prediction_probabilities = model.predict(input_tensor)
        predicted_label = np.argmax(prediction_probabilities)
        
        print("\n--- PREDICTION RESULT ---")
        if predicted_label == 1:
            print("status : The person in the image is wearing a mask ")
        else:
            print("status : The person in the image is not waearing a mask ")  
        print(f"Raw Probabilities [No Mask, Mask] : {prediction_probabilities}")
        
    except Exception as e:
        print(f"Error Processing image at {image_path} : {e}")   
        


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 604ms/step

--- PREDICTION RESULT ---
status : The person in the image is wearing a mask 
Raw Probabilities [No Mask, Mask] : [[0.4258539  0.64195704]]


In [ ]:
sample_image_path = 'test_face.jpg'
predict_face(sample_image_path, model)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step

--- PREDICTION RESULT ---
status : The person in the image is wearing a mask 
Raw Probabilities [No Mask, Mask] : [[0.4258539  0.64195704]]


In [ ]:
model.save('face_mask_model.keras')